In [115]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import os
from rag_helper import RAGBase

In [116]:
instructions = """
You're a course teaching assistant.

Use the search function whenever you need information
from the course FAQ.

If the first search doesn't help,
try another search with different keywords.

Answer only using information from the FAQ.
""".strip()

In [117]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [118]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [119]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    id_field="doc_id",      # <-- use the existing id so if it's re-run, it will update existing records instead of creating duplicates
    db_path='faq.db')

In [120]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [121]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [122]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [123]:
llm_client = OpenAIClient(
    model="openai/gpt-oss-120b",
    client=openai_client
)

In [124]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
    )

In [125]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


/workspaces/llm-zoomcamp-2026-code/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [126]:
result.cost

In [127]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\n\nUse the search function whenever you need information\nfrom the course FAQ.\n\nIf the first search doesn't help,\ntry another search with different keywords.\n\nAnswer only using information from the FAQ.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kw98khm2fz2bmfdyh2m8dn42', summary=[], type='reasoning', content=[Content(text='We need to answer using FAQ. Need to search.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"run Olama locally"}', call_id='fc_523e7970-347d-4fa4-95ee-c2e85592b81e', name='search', type='function_call', id='fc_523e7970-347d-4fa4-95ee-c2e85592b81e', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_523e7970-347d-4fa4-95ee-c2e85592b81e',
  'output': '[\n  {\n    "id":

In [128]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


...

-> Response received


-> Response received


Chat ended.


/workspaces/llm-zoomcamp-2026-code/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:184: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\n\nUse the search function whenever you need information\nfrom the course FAQ.\n\nIf the first search doesn't help,\ntry another search with different keywords.\n\nAnswer only using information from the FAQ.", role='developer', phase=None, type=None), EasyInputMessage(content='', role='user', phase=None, type=None), ResponseReasoningItem(id='resp_01kw98kr2xfx8ayzx20ztf222z', summary=[], type='reasoning', content=[Content(text='We need to answer a question (not yet given). The system says we are a course teaching assistant. Must use search function to find info from FAQ. The user hasn\'t asked yet. Probably the next message will be a question. Wait we are supposed to answer now? There\'s no user question. Possibly the user will ask something next; we need to respond? The instruction says answer only using info from FAQ. Since no question, maybe we need to wait. Probably we should respond that we are r